# QAOA on Max-Cut (primer, project version)

Same as `reference/upstream/qaoa_primer.ipynb`, with code in `src/maxcut.py`.
**Two TODOs in `src/maxcut.py`:** the cost layer in `qaoa_circuit`, and the optimizer in `optimize`. About 45 minutes.

In [ ]:
# Pick the qBraid-SDK (v0.12.2) kernel in qBraid Lab. Locally, any env with requirements.txt works.
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
sys.path.insert(0, str(ROOT))

try:
    import qiskit_aer  # noqa: F401
except ImportError:
    %pip install -q qiskit-aer==0.18.2

%load_ext autoreload
%autoreload 2

import time
import numpy as np
from src.maxcut import (EDGES_SMALL, N_SMALL, EDGES_LARGE, N_LARGE, cut_value, qaoa_circuit,
                        expected_cut, optimize, read_out, brute_force, random_average_cut)

n, edges = N_SMALL, EDGES_SMALL    # Part 1: 6-node grid
p = 1                              # QAOA layers; try 1, then 2

In [ ]:
result = optimize(n, edges, p)     # TODO (cost layer + optimizer) in src/maxcut.py
best_bits, group_a, group_b, best_cut, avg = read_out(result.x, n, edges, p)
print("Group A:", group_a)
print("Group B:", group_b)
print("Cut value:", best_cut, "(compare with the brute-force best below)")
print(f"Average cut per shot: {avg:.2f}   random guessing: {random_average_cut(edges):.1f}")

In [ ]:
print(f"Best possible cut (classical, {n} nodes):", brute_force(n, edges)[0])

### Optional: 12-node scale-up (Part 2)
Switch to the 12-node grid, re-run the two cells above, then the GPU cell.

In [ ]:
n, edges = N_LARGE, EDGES_LARGE
print(f"Now on the {n}-node grid ({len(edges)} lines). Re-run the optimizer and read-out cells.")

In [ ]:
RUN_ON_GPU = False    # free up to a daily limit, then 5 credits / minute
from qiskit import transpile
from src.maxcut import simulator
qc = qaoa_circuit(result.x, n, edges, p)
t0 = time.time(); aer_counts = simulator().run(transpile(qc, simulator()), shots=2048).result().get_counts(); aer_secs = time.time() - t0
if RUN_ON_GPU:
    from qbraid.runtime import QbraidProvider
    from src.config import DEVICES
    device = QbraidProvider().get_device(DEVICES["gpu-sv"])
    t0 = time.time(); job = device.run(qc, shots=2048); gpu_counts = job.result().data.get_counts(); gpu_secs = time.time() - t0
    print("Job ID:", job.id)
    for name, c in [("Aer, here", aer_counts), ("qBraid GPU simulator", gpu_counts)]:
        print(f"{name}: average cut per shot {sum(cut_value(bs[::-1], edges) * k for bs, k in c.items()) / sum(c.values()):.2f}")
    print(f"{n} qubits, 2,048 shots: Aer {aer_secs:.2f} s; GPU {gpu_secs:.0f} s round trip")
else:
    print(f"Aer ran the {n}-qubit circuit (2,048 shots) in {aer_secs:.2f} s.")

## What to have ready (milestone 3)
- The 6-node partition and its cut value, next to the brute-force best.
- Average cut per shot vs random guessing (half the lines), and why the average is the fairer measure.
- One sentence: what the cost layer, mixer and optimizer each do, and how the Max-Cut cost relates to the UC QUBO.
- Optional: 12-node result with Aer vs GPU timing.